In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Exercise merge
* From the prevous exercise we've learned how to combine the information from two data tables.
* Using the learned approach, add the combine the information from the two tables into one, where all patient informaiton (including group name) is available.
* Before making any decisions inspect the two dataframes. Do you notice something that one should pay attention to?
* Which column(s) are you going to use to merge the two dataframes?

In [ ]:
# load the two dataframes

df_patients = pd.read_csv('../../data/predimed_records.csv')

df_groups = pd.read_csv('../../data/predimed_mapping.csv')

In [ ]:
# solution
# inspect teh dataframes

print(len(df_patients))
print(len(df_groups))

In [ ]:
# solution

# merge the two dataframes
df_patients_with_groups = df_patients.merge(df_groups, on = ['patient-id', 'location-id'], how = 'left')

In [ ]:
df_patients_with_groups.group.unique()

How can I change the merge command so that I do not get any missing data?

# Exercise on split-apply-combine: compute summary statistics per group

In the previous exercise we combined two tables with a *join*. Now we take the
joined table and compute a summary statistic *per group* using the
split-apply-combine pattern:

1. **Split** the rows into groups (here, by diet group).
2. **Apply** a function to each group (here, sum the cardiovascular events).
3. **Combine** the per-group results into a single table.

In [ ]:
# df_patients = pd.read_csv('processed_data_predimed.csv')
df_patients = df_patients_with_groups

In [ ]:
df_patients.shape

In [ ]:
df_patients.head()

## Question: is the mediterranean diet associated with less cardiovascular events?

The column `event` contains `Yes` or `No`, indicating whether a patient had a cardiovascular event. 
The column `group` contains which diet the patient followed.

To make the sums easier, we first convert `event` to a binary column (1 for `Yes`, 0 for `No`).

In [ ]:
df_patients['event_int'] = df_patients['event'].map({'Yes': 1, 'No': 0})

Now to answer the question we want the total number of events *per diet group*.

### Naive solution: nested for-loops

A naive way of doing this is as follows. For every group, go through all the rows and add up the events that belong to that group. 

In [ ]:
# just for showing how slow this is. Super slow

events_nested = {}
for g in df_patients['group'].unique():                                 
    total = 0
    for event, group_label in zip(df_patients['event'], df_patients['group']):   
        if group_label == g:
            total += event                                     
    events_nested[g] = total                                  

events_nested

- **Q:** Check out the code above. What is this implementation doing?

In [ ]:
# answer: 
# for every group, it goes through all the rows and add up the events that belong to that group. 

- **Q:** What is the time complexity of this implementation?

In [ ]:
# answer:
# n * g ~~ O(n*g)   with g the number of groups and n the number of rows

Let us measure how long this takes:

In [ ]:
%%timeit
# solution
events_nested = {}
for g in df_patients['group'].unique():
    total = 0
    for event, group_label in zip(df_patients['event'], df_patients['group']):
        if group_label == g:
            total += event
    events_nested[g] = total

### A second solution: iterating through rows

We go through the rows once, and for each row we look up which group it belongs to and add its event to that
group's running total. We keep the running totals in a dictionary.

In [ ]:
# again just for visualizing how slow it is

events_rows = {}
for i, row in df_patients.iterrows():
    g = row['group']
    e = row['event'] # 1 or 0
    if g not in events_rows:
        events_rows[g] = 0
    events_rows[g] += e

events_rows

Let us measure how long the single pass takes:

In [ ]:
%%timeit

events_rows = {}
for i, row in df_patients.iterrows():
    if row['group'] not in events_rows:
        events_rows[row['group']] = 0
    events_rows[row['group']] += row['event']

Same questions for this implementation.

- What is this implementation doing? and 
- What is the time complexity of this implementation?

In [ ]:
# answer:
# it goes through the rows only ONCE. For each row it looks up the group in a
# dictionary and adds the event to that group's running total.
# A dictionary is a hash table, so the look-up and the update cost O(1) on average.
# One pass of n rows, doing O(1) work each --> O(n), independent of g.
#
# So this implementation is asymptotically BETTER than the nested loops, O(n) vs O(n*g).
# And yet it is ~100x SLOWER!  Why? See the discussion at the end of the notebook.

### Exercise: write the fast, one-line version

- Write a single line command that does the same computation with a `split-apply-combine`.
- Check that you get the same numbers as the loops above.
- Time it 

In [ ]:
# solution - optimal
events_groupby = df_patients.groupby('group')['event'].sum()
events_groupby

In [ ]:
%%timeit
# solution
df_patients.groupby('group')['event'].sum()

## Answering the question

There were not the same number of patients in each group, so to compare fairly we look at the *percentage* of events per group. 
*Each of these is a one-line split-apply-combine.*

- Print the number of patients per group

In [ ]:
# solution - number of patients per group
df_patients.groupby('group')['event'].count()

- Calculate the percentage of events per group. 


Hint: the mean of a 0/1 column is the fraction of ones, so multiply by 100 for a percentage

In [ ]:
# solution - percentage of events per group
df_patients.groupby('group')['event'].mean() * 100

The control group had a higher percentage of events than the two mediterranean
diet groups.

## Further explanation about time complexities

Above we had three implementations:

- Nested for-loops: `O(n * g)`
- Loop over rows with `iterrows`: `O(n)`. We walk through the `n` rows exactly once.
- `groupby`: `O(n)`, uses the same idea as iterrows().

Although `iterrows` and `groupby` both have O(n) complexity, `iterrows` was much slower. Why?


In every iteration, `iterrows` creates a new pandas `Series` object for each row and needs to copy the values into it. This cost time.

`groupby`, on the other hand, never leaves compiled code. It runs everything inside pandas' C implementation, walking contiguous typed arrays rather than executing Python bytecode and allocating objects per row. So although both are O(n), `groupby` is much faster.

The takeaway message is: 
- use built-in pandas pipelines to analyse your data.
- do not iterate through rows. If you have a for loop, you're doing it wrong.
